# Default unloading

In [1]:
import json
import matplotlib.pyplot as plt
import numpy as np
import os

from SIMULATION_library.fch_setup import simulation
from SIMULATION_library import simulator_utils
from GSA_library import ionic_output, file_utils

In [2]:
folder_experiment_name          = "Jose/scenarios/YCBD3S9F"


# Probably won't change

username                        = "<archer2_username>"
path2carputils                  = "/work/e348/e348/shared/carputils/"
carp_config_file                = "/work/e348/e348/shared/software/carpentry-system-petsc/carp.conf"
archer2_config_file             = None
path2unloading                  = f"/work/e348/e348/{username}/UNLOADING_library/"
env_folder                      = f"{path2unloading}/venv_UNLOADING_library/"
python_script_path_archer2      = f"{path2unloading}/run.py"

# Particular experiment

harddrive = "/scratch-nvme/e348/e348"
platform = "archer2"
folder_experiment_name_archer          = f"Jose/scenarios///YCBD3S9F"
tags_setup_file_path_archer2    = f"/{harddrive}/{username}/{folder_experiment_name_archer}/json_files/tags_lvrv_fch.json"
general_setup_file_path_archer2 = f"/{harddrive}/{username}/{folder_experiment_name_archer}/json_files/{platform}_setup.json"
states_folder = f"/work/e348/e348/{username}/{folder_experiment_name}/states" # put here the .sv files on the hpc


# local_mesh_folder = f"/media/croderog/SeagateExpansionDrive/{folder_experiment_name}"
local_hard_drive = "/media/croderog/Bob"
local_mesh_folder = f"/{local_hard_drive}/{folder_experiment_name}"
datafolder        = f"{local_mesh_folder}/data"
json_paramfolder  = f"{local_mesh_folder}/json_files"
slrm_folder  = f"{local_mesh_folder}/slrm"
general_setup_file_path_local = f"{json_paramfolder}/{platform}_setup.json"

In [ ]:
cases = os.listdir(f"{local_hard_drive}/{folder_experiment_name}") 

cmd = f"python3 ../simulation_toolbox/write_unloading_scripts.py"
cmd += f" --setup_file {general_setup_file_path_local}" 
cmd += f" --paramfolder {json_paramfolder}" 
cmd += f" --slrmfolder {slrm_folder}" 
cmd += f" --HPC_tags_file {tags_setup_file_path_archer2}" 
cmd += f" --HPC_setup_file {general_setup_file_path_archer2}"
cmd += f" --HPC_path_to_unloading_library {path2unloading}" 
cmd += f" --HPC_env_folder {env_folder}"
cmd += f" --python_script_path_archer2 {python_script_path_archer2}"
cmd += f" --default"

os.system(cmd)

# Default cycle simulation

In [ ]:
local_hard_drive = "/media/croderog/Bob"

cases = os.listdir(f"{local_hard_drive}/Jose/scenarios/") 

for case in cases:

	folder_experiment_name          = f"Jose/scenarios/{case}"


	# Probably won't change

	username                        = "<archer2_username>"
	path2carputils                  = "/work/e348/e348/shared/carputils/"
	carp_config_file                = "/work/e348/e348/shared/software/carpentry-system-petsc/carp.conf"
	archer2_config_file             = None
	path2unloading                  = f"/work/e348/e348/{username}/UNLOADING_library/"
	env_folder                      = f"{path2unloading}/venv_UNLOADING_library/"
	python_script_path_archer2      = f"{path2unloading}/run.py"

	# Particular experiment

	harddrive = "/scratch-nvme/e348/e348"
	platform = "archer2"
	folder_experiment_name_archer          = f"Jose/scenarios///YCBD3S9F"
	tags_setup_file_path_archer2    = f"/{harddrive}/{username}/{folder_experiment_name_archer}/json_files/tags_lvrv_fch.json"
	general_setup_file_path_archer2 = f"/{harddrive}/{username}/{folder_experiment_name_archer}/json_files/{platform}_setup.json"
	states_folder = f"/work/e348/e348/{username}/{folder_experiment_name_archer}/states" # put here the .sv files on the hpc


	# local_mesh_folder = f"/media/croderog/SeagateExpansionDrive/{folder_experiment_name}"
	local_mesh_folder = f"/{local_hard_drive}/{folder_experiment_name}"
	datafolder        = f"{local_mesh_folder}/data"
	json_paramfolder  = f"{local_mesh_folder}/json_files"
	slrm_folder  = f"{local_mesh_folder}/slrm"
	general_setup_file_path_local = f"{json_paramfolder}/{platform}_setup.json"

	sim_setup = simulation()

	sim_setup.load(general_setup_file_path_local)

	original_meshname = sim_setup.meshname

	where_to_save_param_string = f"{local_mesh_folder}/SS/param/"
	os.system("rm -r "+where_to_save_param_string)

	print("Saving the parameter files for the cell simulations in "+where_to_save_param_string+"...")


	sim_setup.meshname = "/unloaded/"+original_meshname+"_unloaded_default"
	sim_setup.testname = "cycle_default"

	sim_setup.torord_init_file = f"{states_folder}/ToRORd_dynCl/default_ToRORd_dynCl_LandHumanStress.sv"
	sim_setup.torord_rv_init_file = f"{states_folder}/ToRORd_dynCl_rv/default_ToRORd_dynCl_rv_LandHumanStress.sv"
	sim_setup.courtemanche_init_file = f"{states_folder}/JB_COURTEMANCHE/default_JB_COURTEMANCHE_LandHumanStress.sv" 

	simulator_utils.write_simulation_script(f"{json_paramfolder}/default.json",
											f"{local_mesh_folder}/json_files/clinical_data_GENERIC.json",
											f"{local_mesh_folder}/json_files/tags_lvrv_fch.json",
											f"{slrm_folder}/cycle_default.slrm",
											sim_setup,
											postprocessing=False,
											adapt_tubeArea=True,
											save_param_string=f"{where_to_save_param_string}/default_")


# Default cell simulation

In [ ]:
def generate_default_bench_script(
						  BCL,
						  NBEATS,
						  basefolder,
						  strain,
						  chamber,
						  contraction_model,
						  suffix):

	if basefolder[-1]=="/":
		basefolder = basefolder[:-1]
	
	if chamber=="LV":
		ionic_model = "ToRORd_dynCl"
	elif chamber=="RV":
		ionic_model = "ToRORd_dynCl"
	elif chamber=="atria":
		ionic_model = "JB_COURTEMANCHE"
	else:
		raise Exception("Unsupported chamber - pick LV, RV or atria")

	f = open(f"{basefolder}/run_{ionic_model}{suffix}.sh","w")

	f.write("#!/bin/bash\n")

	os.system("mkdir "+basefolder+"/"+ionic_model+suffix+"/")

	f.write("\n")
	
	DURATION  = NBEATS*BCL

	cmd=["cmd="+'"'+"bench.pt"]
	cmd += ["--numstim",str(NBEATS)+"\n"]
	cmd += ["--bcl",str(BCL)+"\n"]
	cmd += ["--past-stim",str(BCL)+"\n"]
	cmd += ["--imp",ionic_model+"\n"]
	cmd += ["--dt","0.02"+"\n"]
	cmd += ["--stim-curr","60.0"+"\n"]
	cmd += ["--dt-out","1.0"+"\n"]

	if contraction_model=="Land":
		cmd += ["--plug-in","LandHumanStress"+"\n"]
		cmd += ["--save-ini-file","${output}/default_"+ionic_model+suffix+"_LandHumanStress.sv"+"\n"]
	elif contraction_model=="tanh":
		cmd += ["--plug-in","TanhStress"+"\n"]
		cmd += ["--save-ini-file","${output}/default_"+ionic_model+suffix+"_TanhStress"+suffix+".sv"+"\n"]

	cmd += ["--strain",str(strain)+"\n"]
	cmd += ["--strain-rate","0.0"+"\n"]
	cmd += ["--strain-time","0.0"+"\n"]
	cmd += ["--strain-dur",str(DURATION)+"\n"]
	cmd += ["--imp-par","${pp_ionic}"+"\n"]
	cmd += ["--plug-par","${pp_land}"+"\n"]
	cmd += ["--save-ini-time ",str(DURATION)+"\n"]

	cmd += ["--fout=${output}/"+ionic_model+"\n"]
	cmd += ["--plug-sv-dump=Tension"+"\n"]
	cmd += ["--imp-sv-dump=Ca_i"+"\n"]

	cmd_str = " ".join(cmd)
	cmd_final = cmd_str+" --bin\n --no-trace\n "+'"\n'

	# --------------------------------------------------
# first loop 

	f.write("\n")
	f.write("\n")

	f.write("pp_ionic=$(cat "+basefolder+"/param/default_param_"+ionic_model+".txt)\n")
	f.write("pp_land=$(cat "+basefolder+"/param/default_param_"+ionic_model+"_"+contraction_model+suffix+".txt)\n")	

	f.write("\n")	

	f.write("output="+basefolder+"/"+ionic_model+suffix+"/default\n")
	f.write("mkdir ${output}\n")	

	f.write("\n")
	f.write(cmd_final)
	f.write("\n")
	f.write("echo $cmd\n")
	f.write("eval $cmd\n")
	f.write("\n")


	f.write("\n")
	f.write("\n")
	f.write("\n")
	f.write("\n")

	print(f"Written in {basefolder}/run_{ionic_model}{suffix}.sh")

def bin_to_dat_own(filename,
			   BCL,
			   Nbeats,
			   output_filename,
			   cleanup=False,
			   save_Nbeats=1):
	
	if not os.path.exists(filename):
		raise Exception('Cannot find '+filename+'.')	

	s = np.fromfile(filename, dtype="float64") 

	if save_Nbeats is None:
		save_Nbeats = Nbeats
	print(f"s.shape[0] is {s.shape[0]} and should be {Nbeats*BCL+2}")
	print(f"Then {s.shape[0] == Nbeats*BCL+2}")
	if s.shape[0] == (Nbeats*BCL+2):
		print(f"Writing in {output_filename}")
		f = open(output_filename,"w")
		f.write("\n")				

		for i in range(BCL*(Nbeats-save_Nbeats),BCL*Nbeats-1):
			f.write("      ("+str(i)+"): "+str(s[i])+",\n")
		f.write("      ("+str(int(BCL*Nbeats-1))+"): "+str(s[-2]))
		f.close()

	if cleanup:
		os.system("rm "+filename)

def bin_to_dat_folder_default(sim_foldername,
					  BCL,
					  Nbeats,
					  bin_files,
					  output_files,
					  cleanup=False,
					  save_Nbeats=1,
					  suffix=""):



	folder = sim_foldername+"/default/"

	for j,f in enumerate(bin_files):

		filename = folder+f
		print(f"Converting {filename} to {folder}{output_files[j]}")
		bin_to_dat_own(filename,
					BCL,
					Nbeats,
					folder+output_files[j],
					cleanup=cleanup,
					save_Nbeats=save_Nbeats)

	if cleanup:
		os.system("rm "+folder+"*.bin")

def plot_Land_output_default(basefolder,
                     figname,
                     ionic_model,
					 clinical_data,
					 num_beats_limit_cycle,
					 suffix=""):

	bin_files = [f"{ionic_model}.Vm.bin",f"{ionic_model}.Ca_i.bin",f"{ionic_model}.Tension.bin"]
	output_files = [f"{ionic_model}.Vm.dat",f"{ionic_model}.Ca_i.dat",f"{ionic_model}.Tension.dat"]

	basefolder = f"{local_mesh_folder}/SS/{ionic_model}{suffix}/"

		
	bin_to_dat_folder_default(sim_foldername = f"{basefolder}",
						BCL=int(clinical_data["general"]["BCL"]),
						Nbeats=int(num_beats_limit_cycle),
						bin_files=bin_files,
						output_files=output_files,
						cleanup=False)



	ax = plt.figure(figsize=(10,5), constrained_layout=True).subplots(1, 3)


	T_1 = file_utils.read_ionic_output(f"{basefolder}/default/{output_files[2]}")
	Ca_i_1 = file_utils.read_ionic_output(f"{basefolder}/default/{output_files[1]}")
	Vm_1 = file_utils.read_ionic_output(f"{basefolder}/default/{output_files[0]}")
	t_1 = np.arange(0, Ca_i_1.shape[0])

	# Plot the curve
	ax[0].plot(t_1, Ca_i_1, color='#3489eb')
	ax[1].plot(t_1, T_1, color='#3489eb')
	ax[2].plot(t_1, Vm_1, color='#3489eb')

	ax[0].set_xlabel('Time [ms]')
	ax[1].set_xlabel('Time [ms]')

	ax[0].set_ylabel('Ca_i [um]')
	ax[1].set_ylabel('Tension [kPa]')
	ax[2].set_ylabel('Vm')


	plt.savefig(figname, dpi=100)

In [ ]:
with open(f"{local_mesh_folder}/json_files/clinical_data_GENERIC.json","r") as f:
    clinical_data = json.load(f)

num_beats_limit_cycle  = 500


## Ventricles: ToRORd_dynCl + LandHumanStress

In [ ]:


ionic_model = "ToRORd_dynCl"


generate_default_bench_script(
					  BCL=clinical_data["general"]["BCL"],						    	# BCL
					  NBEATS=num_beats_limit_cycle,							# NBEATS
					  basefolder=f"{local_mesh_folder}/SS/",				# basefolder containing the param folder
					  strain=0.0,								# strain
					  chamber="LV",		 						# chamber = LV, RV or atria
					  contraction_model=sim_setup.contraction_model, 		# contraction model
					  suffix="")

os.system(f"bash {local_mesh_folder}/SS/run_ToRORd_dynCl.sh")

plot_Land_output_default(basefolder = f"{local_mesh_folder}/SS/{ionic_model}/",
                     figname = f"{local_mesh_folder}/SS/{ionic_model}/{ionic_model}_output.png",
                     ionic_model=ionic_model,
					 clinical_data=clinical_data,
					 num_beats_limit_cycle=num_beats_limit_cycle)




# ionic_model = "ToRORd_dynCl"
# suffix = "_rv"


# generate_default_bench_script(
# 					  BCL=clinical_data["general"]["BCL"],						    	# BCL
# 					  NBEATS=num_beats_limit_cycle,							# NBEATS
# 					  basefolder=f"{local_mesh_folder}/SS/",				# basefolder containing the param folder
# 					  strain=0.0,								# strain
# 					  chamber="RV",		 						# chamber = LV, RV or atria
# 					  contraction_model=sim_setup.contraction_model, 		# contraction model
# 					  suffix="_rv")

# os.system(f"bash {local_mesh_folder}/SS/run_ToRORd_dynCl_rv.sh")

# plot_Land_output_default(basefolder = f"{local_mesh_folder}/SS/{ionic_model}{suffix}/",
#                      figname = f"{local_mesh_folder}/SS/{ionic_model}{suffix}/{ionic_model}{suffix}_output.png",
#                      ionic_model=ionic_model,
# 					 clinical_data=clinical_data,
# 					 num_beats_limit_cycle=num_beats_limit_cycle,
# 					 suffix=suffix)


# ionic_model = "JB_COURTEMANCHE"


# generate_default_bench_script(
# 					  BCL=clinical_data["general"]["BCL"],						    	# BCL
# 					  NBEATS=num_beats_limit_cycle,							# NBEATS
# 					  basefolder=f"{local_mesh_folder}/SS/",				# basefolder containing the param folder
# 					  strain=0.0,								# strain
# 					  chamber="atria",		 						# chamber = LV, RV or atria
# 					  contraction_model=sim_setup.contraction_model, 		# contraction model
# 					  suffix="")

# os.system(f"bash {local_mesh_folder}/SS/run_JB_COURTEMANCHE.sh")

# plot_Land_output_default(basefolder = f"{local_mesh_folder}/SS/{ionic_model}/",
#                      figname = f"{local_mesh_folder}/SS/{ionic_model}/{ionic_model}_output.png",
#                      ionic_model=ionic_model,
# 					 clinical_data=clinical_data,
# 					 num_beats_limit_cycle=num_beats_limit_cycle)


# Inflation

In [3]:
fields = ['mechanics']
mechDT = 1.0
N=1
# -------------------------------------------------------------------------
local_hard_drive = "/media/croderog/Bob"
folder_experiment_name = "Jose/scenarios/"

cases = os.listdir(f"{local_hard_drive}/{folder_experiment_name}") 


for case in cases:
    base_folder_local = f"{local_hard_drive}/{folder_experiment_name}/{case}"
    data_folder = f"{base_folder_local}/data"

    cmd = f"python ../simulation_toolbox/write_inflation_scripts.py"
    cmd += f" --platform {platform}"
    cmd += f" --user {username}"
    cmd += f" --paramfolder {base_folder_local}/json_files/"
    cmd += f" --slrmfolder {base_folder_local}/slrm/"
    cmd += f" --defaultfile {base_folder_local}/json_files/default.json"  #### MAKE SURE THIS IS OK
    cmd += f" --clinical_data {base_folder_local}/json_files/clinical_data_GENERIC.json"
    cmd += f" --tags {base_folder_local}/json_files/tags_lvrv_fch.json"
    cmd += f" --setup_file {base_folder_local}/json_files/{platform}_setup.json"
    cmd += f" --mechDT {mechDT}"
    cmd += f" --default"
    cmd += f" --datafolder {data_folder}"
    os.system(cmd)

Loading /media/croderog/Bob/Jose/scenarios//OHV835G6/json_files/archer2_setup.json...
Done.
Trace file created at: /media/croderog/Bob/Jose/scenarios//OHV835G6/data/inflation_trace.trc
----------------------------
Ionic model @ passive:
impID : 0
name : passive
ionic_model : PASSIVE
IDs_list : [1, 2, 3, 4, 25, 28, 29, 26, 27, 5, 6, 18, 19, 20, 21, 22, 23, 24, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
param : None
plugin : None
plug_param : None
im_sv_init : None
----------------------------
----------------------------
Mech region @ lv:
name : lv
mregID : 0
IDs_list : [1, 25, 29, 27]
mtype : 9
params : kappa=1000.0,a=1.7,b_f=8.0,b_fs=4.0,b_t=3.0
----------------------------
----------------------------
Mech region @ rv:
name : rv
mregID : 1
IDs_list : [2, 28]
mtype : 9
params : kappa=1000.0,a=2.55,b_f=8.0,b_fs=4.0,b_t=3.0
----------------------------
----------------------------
Mech region @ atria:
name : atria
mregID : 2
IDs_list : [3, 4, 26]
mtype : 9
params : kappa=1000.0,a=3.4,b_f=

Traceback (most recent call last):
  File "/home/croderog/Desktop/IC_projects/simulation_toolbox/notebooks/../simulation_toolbox/write_inflation_scripts.py", line 230, in <module>
    main(args)
  File "/home/croderog/Desktop/IC_projects/simulation_toolbox/notebooks/../simulation_toolbox/write_inflation_scripts.py", line 106, in main
    raise Exception("You need to have a file called "+setup_file)
Exception: You need to have a file called /media/croderog/Bob/Jose/scenarios//4ch-cohort/json_files/archer2_setup.json


Loading /media/croderog/Bob/Jose/scenarios//FC14/json_files/archer2_setup.json...
Done.
Trace file created at: /media/croderog/Bob/Jose/scenarios//FC14/data/inflation_trace.trc
----------------------------
Ionic model @ passive:
impID : 0
name : passive
ionic_model : PASSIVE
IDs_list : [1, 2, 3, 4, 25, 28, 29, 26, 27, 5, 6, 18, 19, 20, 21, 22, 23, 24, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
param : None
plugin : None
plug_param : None
im_sv_init : None
----------------------------
----------------------------
Mech region @ lv:
name : lv
mregID : 0
IDs_list : [1, 25, 29, 27]
mtype : 9
params : kappa=1000.0,a=1.7,b_f=8.0,b_fs=4.0,b_t=3.0
----------------------------
----------------------------
Mech region @ rv:
name : rv
mregID : 1
IDs_list : [2, 28]
mtype : 9
params : kappa=1000.0,a=2.55,b_f=8.0,b_fs=4.0,b_t=3.0
----------------------------
----------------------------
Mech region @ atria:
name : atria
mregID : 2
IDs_list : [3, 4, 26]
mtype : 9
params : kappa=1000.0,a=3.4,b_f=16.0,b_f

Traceback (most recent call last):
  File "/home/croderog/Desktop/IC_projects/simulation_toolbox/notebooks/../simulation_toolbox/write_inflation_scripts.py", line 230, in <module>
    main(args)
  File "/home/croderog/Desktop/IC_projects/simulation_toolbox/notebooks/../simulation_toolbox/write_inflation_scripts.py", line 106, in main
    raise Exception("You need to have a file called "+setup_file)
Exception: You need to have a file called /media/croderog/Bob/Jose/scenarios//only_simulations/json_files/archer2_setup.json


Loading /media/croderog/Bob/Jose/scenarios//MHFw8/json_files/archer2_setup.json...
Done.
Trace file created at: /media/croderog/Bob/Jose/scenarios//MHFw8/data/inflation_trace.trc
----------------------------
Ionic model @ passive:
impID : 0
name : passive
ionic_model : PASSIVE
IDs_list : [1, 2, 3, 4, 25, 28, 29, 26, 27, 5, 6, 18, 19, 20, 21, 22, 23, 24, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
param : None
plugin : None
plug_param : None
im_sv_init : None
----------------------------
----------------------------
Mech region @ lv:
name : lv
mregID : 0
IDs_list : [1, 25, 29, 27]
mtype : 9
params : kappa=1000.0,a=1.7,b_f=8.0,b_fs=4.0,b_t=3.0
----------------------------
----------------------------
Mech region @ rv:
name : rv
mregID : 1
IDs_list : [2, 28]
mtype : 9
params : kappa=1000.0,a=2.55,b_f=8.0,b_fs=4.0,b_t=3.0
----------------------------
----------------------------
Mech region @ atria:
name : atria
mregID : 2
IDs_list : [3, 4, 26]
mtype : 9
params : kappa=1000.0,a=3.4,b_f=16.0,b